# 1 — Dispatch and the price

Companion to **section 2** of the note. Everything here is the single-hour
linear program:

$$\min_q \sum_g c_g q_g \quad\text{s.t.}\quad \sum_g q_g = D \;\;(\lambda),
\qquad q_g \le \bar q_g \;\;(\mu_g), \qquad q_g \ge 0.$$

In [ ]:
# Make the note's models importable, wherever you launched Jupyter from.
import sys
from pathlib import Path

here = Path.cwd()
note = next(p for p in [here, *here.parents] if (p / "model" / "dispatch.py").exists())
sys.path.insert(0, str(note))
PROCESSED = note / "data" / "processed"

import pandas as pd
import matplotlib.pyplot as plt

print(f"note root: {note}")

In [ ]:
from model import dispatch
from pipeline.run_dispatch import CAPACITY_MW, LOAD_MW

tech = dispatch.read_tech(PROCESSED / "technology_costs_small.csv")
mc = dispatch.marginal_cost(tech).loc[CAPACITY_MW.index]

## The merit order, drawn

Sort by marginal cost, stack the capacities, and intersect with demand. This
is Figure 2.1 of the note, built from scratch.

In [ ]:
order = mc.sort_values().index
widths = CAPACITY_MW[order].to_numpy()
left = widths.cumsum() - widths

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(left, mc[order], width=widths, align="edge",
       color="#aecde3", edgecolor="white")
for x, w, t in zip(left, widths, order):
    if w > 300:
        ax.text(x + w / 2, mc[t] + 2, tech.loc[t, "label"],
                ha="center", fontsize=7, rotation=90, va="bottom")
ax.axvline(LOAD_MW, color="#D55E00", lw=1.6)
ax.text(LOAD_MW, ax.get_ylim()[1] * 0.9, "  demand", color="#D55E00")
ax.set_xlabel("cumulative capacity (MW)")
ax.set_ylabel("marginal cost (EUR/MWh)")
plt.show()

## The price is a dual variable

Solve, and compare the dual of market clearing with the marginal cost of the
plant the demand line lands on.

In [ ]:
sol = dispatch.solve(tech, CAPACITY_MW, LOAD_MW)
marginal_tech = (sol["generation"][(sol["generation"] > 1e-6) &
                 (sol["generation"] < CAPACITY_MW - 1e-6)])
print("price (dual of balance):", round(sol["price"], 2), "EUR/MWh")
print("partially dispatched   :", list(marginal_tech.index))
print("its marginal cost      :", round(mc[marginal_tech.index[0]], 2))

## A carbon tax re-sorts the merit order

A tax $\tau$ changes each plant's effective cost to
$c_g + \tau e_g$ — nothing else about the problem changes.

In [ ]:
rows = []
for tau in [0, 40, 85, 150]:
    s = dispatch.solve(tech, CAPACITY_MW, LOAD_MW, co2_price=tau)
    rows.append({"tax (EUR/t)": tau,
                 "price (EUR/MWh)": round(s["price"], 1),
                 "emissions (t)": round(s["emissions"]),
                 "coal dispatched (MW)": round(s["generation"].get("coal_chp", 0))})
pd.DataFrame(rows)

Notice the pass-through: the price rises by $\tau \cdot e_m$ for the
*marginal* technology, not by the average carbon content of what is consumed.

## The cap, and the marginal abatement cost curve

Now fix the quantity instead of the price. The dual of the cap is an
endogenous carbon price — and sweeping the cap traces a marginal abatement
cost curve that the model **derives** rather than assumes. This is Figure 2.4.

In [ ]:
import numpy as np

base = dispatch.solve(tech, CAPACITY_MW, LOAD_MW)
floor_sol = dispatch.solve(tech, CAPACITY_MW, LOAD_MW, co2_price=10_000)
m0, floor = base["emissions"], floor_sol["emissions"]

caps = np.linspace(floor * 1.001, m0, 120)
sigma = [dispatch.solve(tech, CAPACITY_MW, LOAD_MW, co2_cap=c)["co2_shadow_price"]
         for c in caps]

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.step(m0 - caps, sigma, where="post", color="#0072B2", lw=1.8)
ax.set_xlabel("abatement  A = M0 - M  (t CO2)")
ax.set_ylabel("marginal abatement cost (EUR/t)")
plt.show()

Every flat step is one abatement margin — a named pair of technologies
swapping places in the merit order. Every vertical jump is a margin being
exhausted.

## Your turn

1. Which two technologies produce the first (cheapest) step? Find them by
   comparing dispatch just either side of the first jump.
2. On a vertical segment, a whole interval of carbon prices supports the same
   emissions. Pick a cap on a jump and confirm that a *tax* anywhere in that
   interval gives the same dispatch.
3. The note's section 2.4 adds load shedding at a value of lost load. Add a
   "shedding" pseudo-generator with a marginal cost of 10,000 EUR/MWh and
   unlimited capacity. What is the price when demand exceeds firm capacity?

In [ ]:
# Try it here.